# Market Replay and Slippage

Generate synthetic events, replay them sequentially, inspect microstructure time series, and simulate execution fills without lookahead.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
from lob_engine.core.orders import Side
from lob_engine.execution import ParentOrder, TWAPExecutor
from lob_engine.execution.base import summarize_fills
from lob_engine.simulation.fill_simulator import simulate_child_order_fills
from lob_engine.simulation.market_generator import SyntheticMarketConfig, generate_market_events
from lob_engine.simulation.market_replay import MarketReplay

market_events = generate_market_events(SyntheticMarketConfig(num_events=2000, seed=42))
replay = MarketReplay().replay(market_events)
replay.snapshots.tail()

In [ ]:
fig = px.line(replay.snapshots, x='timestamp', y=['mid_price', 'spread'], title='Replay Mid Price and Spread')
fig.show()

In [ ]:
parent = ParentOrder('P-REPLAY', Side.BUY, 1000, replay.snapshots['timestamp'].min(), replay.snapshots['timestamp'].max(), arrival_price=100.0)
schedule = TWAPExecutor(parent, slices=10).build_schedule()
fills = simulate_child_order_fills(schedule, replay.snapshots, Side.BUY)
fills.head()

In [ ]:
summary = summarize_fills(parent, fills.assign(market_volume=fills['filled_quantity'] / 0.1))
summary